# Phase 4 — Full Multi-Task Scaffold (URI-CADS)

Extends Phase 2/3's architecture with two new heads, using the `parse_local_lesions()` / `bbox_to_coarse_mask()` functions built in Phase 1:

- **Head C — coarse lesion segmentation**: a single binary mask per image (union of all lesion boxes), predicting *where* any abnormality is. This is your "a square works too" novelty piece.
- **Head D — local pathology classification**: multi-label vector over the 9 raw pathology IDs, predicting *what* abnormality types are present anywhere in the image.

Loss is still a **fixed, unweighted sum** across all 5 terms (diagnosis + global pathology + kidney seg + lesion mask + local pathology) -- Phase 6 is where these weights get optimized, not here.

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/URI-CADS dataset'
DRIVE_ZIP_PATH = f'{DRIVE_PROJECT_ROOT}/uri_cads_archive.zip'
DRIVE_CHECKPOINTS_PATH = f'{DRIVE_PROJECT_ROOT}/checkpoints'
MANIFEST_SAVE_PATH = f'{DRIVE_PROJECT_ROOT}/manifest_with_splits.pkl'

LOCAL_ZIP_PATH = '/content/uri_cads_archive.zip'
LOCAL_DATA_DIR = '/content/uri_cads_data'
LOCAL_DATASET_DIR = os.path.join(LOCAL_DATA_DIR, 'bbdd_kidney')

os.makedirs(DRIVE_CHECKPOINTS_PATH, exist_ok=True)
print('Zip exists on Drive:', os.path.exists(DRIVE_ZIP_PATH))
print('Saved manifest exists:', os.path.exists(MANIFEST_SAVE_PATH))

Zip exists on Drive: True
Saved manifest exists: True


In [4]:
import shutil, zipfile, time

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
t0 = time.time()
shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
    zf.extractall(LOCAL_DATA_DIR)
print(f'Copied + unzipped in {time.time()-t0:.1f}s')
print('Contents:', os.listdir(LOCAL_DATASET_DIR))

Copied + unzipped in 2.8s
Contents: ['images', 'masks', 'labels']


In [5]:
import pandas as pd

manifest = pd.read_pickle(MANIFEST_SAVE_PATH)
print(manifest['split'].value_counts())

split
train    1389
val       298
test      298
Name: count, dtype: int64


In [6]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -- switch Runtime > Change runtime type > GPU before continuing')

True Tesla T4


## Lesion parsing (from Phase 1, duplicated here for a self-contained notebook)

In [9]:
import numpy as np

NUM_LOCAL_PATHOLOGY_IDS = 9  # confirmed range 1-9 from Phase 1 exploration


def parse_local_lesions(extra_lines):
    lesions = []
    for line in extra_lines:
        if not line.startswith('Bounding box'):
            continue
        _, values = line.split(':', 1)
        nums = [int(v) for v in values.split()]
        if len(nums) != 5:
            continue
        x, y, w, h, pathology_id = nums
        lesions.append({'bbox': (x, y, x + w, y + h), 'pathology_id': pathology_id})
    return lesions


def bbox_to_coarse_mask(bbox, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    x_min, y_min, x_max, y_max = [int(round(v)) for v in bbox]
    mask[y_min:y_max, x_min:x_max] = 1
    return mask

## Dataset class -- adds lesion union mask (Head C target) and local pathology multi-label vector (Head D target)

In [10]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import scipy.io

IMG_SIZE = (512, 512)


def load_kidney_mask(mat_path):
    data = scipy.io.loadmat(mat_path)
    mask = data['mask']
    return np.asarray(mask).astype(np.uint8)


class URICADSMultiTaskDataset(Dataset):
    def __init__(self, manifest_df, img_size=IMG_SIZE):
        self.manifest = manifest_df.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        target_h, target_w = self.img_size

        image = Image.open(row['image_path']).convert('RGB')
        orig_width, orig_height = image.size
        image = image.resize((target_w, target_h), Image.BILINEAR)
        image = np.array(image).astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5

        kidney_mask = load_kidney_mask(row['mask_path'])
        kidney_mask = np.array(Image.fromarray(kidney_mask).resize((target_w, target_h), Image.NEAREST))

        global_diagnosis = int(row['global_diagnosis'])
        global_pathologies = torch.tensor(row['global_pathologies'], dtype=torch.float32)

        # --- lesion union mask (Head C target) + local pathology multi-label vector (Head D target) ---
        scale_x = target_w / orig_width
        scale_y = target_h / orig_height
        lesions = parse_local_lesions(row['extra_lines'])

        lesion_union_mask = np.zeros((target_h, target_w), dtype=np.uint8)
        local_pathology_vec = np.zeros(NUM_LOCAL_PATHOLOGY_IDS, dtype=np.float32)
        for lesion in lesions:
            x_min, y_min, x_max, y_max = lesion['bbox']
            scaled_box = (x_min * scale_x, y_min * scale_y, x_max * scale_x, y_max * scale_y)
            box_mask = bbox_to_coarse_mask(scaled_box, target_h, target_w)
            lesion_union_mask = np.maximum(lesion_union_mask, box_mask)
            pid = lesion['pathology_id']
            if 1 <= pid <= NUM_LOCAL_PATHOLOGY_IDS:
                local_pathology_vec[pid - 1] = 1.0  # IDs are 1-indexed

        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        kidney_mask_tensor = torch.from_numpy(kidney_mask).float()
        lesion_mask_tensor = torch.from_numpy(lesion_union_mask).float()

        return {
            'image': image_tensor,
            'kidney_mask': kidney_mask_tensor,
            'global_diagnosis': torch.tensor(global_diagnosis, dtype=torch.float32),
            'global_pathologies': global_pathologies,
            'lesion_mask': lesion_mask_tensor,
            'local_pathologies': torch.from_numpy(local_pathology_vec),
            'case_id': row['case_id'],
        }


train_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'train'])
val_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'val'])
test_ds = URICADSMultiTaskDataset(manifest[manifest['split'] == 'test'])

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
sample = train_ds[0]
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample.items()})

Train: 1389 | Val: 298 | Test: 298
{'image': torch.Size([3, 512, 512]), 'kidney_mask': torch.Size([512, 512]), 'global_diagnosis': torch.Size([]), 'global_pathologies': torch.Size([2]), 'lesion_mask': torch.Size([512, 512]), 'local_pathologies': torch.Size([9]), 'case_id': '10'}


## Model -- shared encoder + 4 heads (diagnosis, global pathology, kidney seg, lesion mask) + local pathology head

In [11]:
import torch.nn as nn
import torchvision

NUM_GLOBAL_PATHOLOGIES = len(manifest.iloc[0]['global_pathologies'])


class MultiTaskNetV2(nn.Module):
    def __init__(self, num_global_pathologies=2, num_local_pathologies=NUM_LOCAL_PATHOLOGY_IDS, pretrained=True):
        super().__init__()
        backbone = torchvision.models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])  # -> B x 512 x H/32 x W/32

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.diagnosis_head = nn.Linear(512, 1)
        self.global_pathology_head = nn.Linear(512, num_global_pathologies)
        self.local_pathology_head = nn.Linear(512, num_local_pathologies)  # Head D

        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            )

        # kidney segmentation decoder (Head A)
        self.kidney_decoder = nn.Sequential(
            up_block(512, 256), up_block(256, 128), up_block(128, 64),
            up_block(64, 32), up_block(32, 16),
            nn.Conv2d(16, 1, kernel_size=1),
        )

        # lesion coarse-segmentation decoder (Head C) -- separate decoder, shares only the encoder
        self.lesion_decoder = nn.Sequential(
            up_block(512, 256), up_block(256, 128), up_block(128, 64),
            up_block(64, 32), up_block(32, 16),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x):
        feat = self.encoder(x)
        pooled = self.global_pool(feat).flatten(1)

        diagnosis_logits = self.diagnosis_head(pooled).squeeze(1)
        global_pathology_logits = self.global_pathology_head(pooled)
        local_pathology_logits = self.local_pathology_head(pooled)

        kidney_seg_logits = self.kidney_decoder(feat).squeeze(1)
        lesion_seg_logits = self.lesion_decoder(feat).squeeze(1)

        return {
            'diagnosis': diagnosis_logits,
            'global_pathology': global_pathology_logits,
            'local_pathology': local_pathology_logits,
            'kidney_seg': kidney_seg_logits,
            'lesion_seg': lesion_seg_logits,
        }


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskNetV2(num_global_pathologies=NUM_GLOBAL_PATHOLOGIES).to(device)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 134MB/s]


## Loss -- fixed, unweighted sum across all 5 terms (Phase 6 optimizes these weights, not here)

In [12]:
def dice_loss(pred_logits, target, eps=1e-6):
    pred = torch.sigmoid(pred_logits)
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


bce_logits = nn.BCEWithLogitsLoss()


def compute_loss(outputs, batch):
    loss_diagnosis = bce_logits(outputs['diagnosis'], batch['global_diagnosis'].to(device))
    loss_global = bce_logits(outputs['global_pathology'], batch['global_pathologies'].to(device))
    loss_local = bce_logits(outputs['local_pathology'], batch['local_pathologies'].to(device))
    loss_kidney_seg = dice_loss(outputs['kidney_seg'], batch['kidney_mask'].to(device))
    loss_lesion_seg = dice_loss(outputs['lesion_seg'], batch['lesion_mask'].to(device))

    # FIXED, EQUAL (unweighted) sum -- matches URI-CADS's own stated design choice, extended
    # to the two new heads. Phase 6 replaces this with optimized weights.
    total_loss = loss_diagnosis + loss_global + loss_local + loss_kidney_seg + loss_lesion_seg

    breakdown = {
        'diagnosis': loss_diagnosis.item(),
        'global_pathology': loss_global.item(),
        'local_pathology': loss_local.item(),
        'kidney_seg': loss_kidney_seg.item(),
        'lesion_seg': loss_lesion_seg.item(),
    }
    return total_loss, breakdown

## Training setup

In [15]:
BATCH_SIZE = 4
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
from tqdm.auto import tqdm


def dice_score(pred_logits, target, eps=1e-6):
    pred = (torch.sigmoid(pred_logits) > 0.5).float()
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    return ((2 * intersection + eps) / (union + eps)).mean().item()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_kidney_dice = 0.0
    total_lesion_dice = 0.0
    n_batches = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in tqdm(loader, desc='train' if train else 'val'):
            images = batch['image'].to(device)
            outputs = model(images)
            loss, _ = compute_loss(outputs, batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_kidney_dice += dice_score(outputs['kidney_seg'], batch['kidney_mask'].to(device))
            total_lesion_dice += dice_score(outputs['lesion_seg'], batch['lesion_mask'].to(device))
            n_batches += 1

    return (total_loss / n_batches, total_kidney_dice / n_batches, total_lesion_dice / n_batches)

## Tiny sanity test -- run BEFORE the full loop

In [ ]:
tiny_train = torch.utils.data.Subset(train_ds, range(20))
tiny_loader = DataLoader(tiny_train, batch_size=4, num_workers=0)
t0 = time.time()
run_epoch(tiny_loader, train=True)
print(f'20 samples took {time.time()-t0:.1f}s')

train:   0%|          | 0/5 [00:00<?, ?it/s]

20 samples took 2.7s


## Full training loop

In [ ]:
best_val_score = 0.0  # tracked as (kidney_dice + lesion_dice) / 2
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_kidney_dice, train_lesion_dice = run_epoch(train_loader, train=True)
    val_loss, val_kidney_dice, val_lesion_dice = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0

    val_score = (val_kidney_dice + val_lesion_dice) / 2

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'train_loss={train_loss:.4f} | '
          f'val_loss={val_loss:.4f} val_kidney_dice={val_kidney_dice:.4f} val_lesion_dice={val_lesion_dice:.4f} | '
          f'{elapsed:.1f}s')

    if val_score > best_val_score:
        best_val_score = val_score
        checkpoint_path = os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase4_multitask_best.pt')
        torch.save(model.state_dict(), checkpoint_path)
        print(f'  -> new best combined dice={val_score:.4f}, checkpoint saved to Drive')

train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch  1/15 | train_loss=3.0468 | val_loss=2.8722 val_kidney_dice=0.6764 val_lesion_dice=0.1612 | 79.4s
  -> new best combined dice=0.4188, checkpoint saved to Drive


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

...

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 14/15 | train_loss=1.1667 | val_loss=2.0615 val_kidney_dice=0.8847 val_lesion_dice=0.3322 | 83.9s
  -> new best combined dice=0.6085, checkpoint saved to Drive


train:   0%|          | 0/348 [00:00<?, ?it/s]

val:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 15/15 | train_loss=1.1647 | val_loss=2.0492 val_kidney_dice=0.8932 val_lesion_dice=0.3465 | 84.7s
  -> new best combined dice=0.6198, checkpoint saved to Drive


## Test-set evaluation

In [16]:
from sklearn.metrics import roc_auc_score

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.load_state_dict(torch.load(os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase4_multitask_best.pt')))
model.eval()

all_diag_probs, all_diag_targets = [], []
all_local_probs, all_local_targets = [], []
all_kidney_dice, all_lesion_dice = [], []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        outputs = model(images)

        all_diag_probs.extend(torch.sigmoid(outputs['diagnosis']).cpu().numpy())
        all_diag_targets.extend(batch['global_diagnosis'].numpy())

        all_local_probs.append(torch.sigmoid(outputs['local_pathology']).cpu().numpy())
        all_local_targets.append(batch['local_pathologies'].numpy())

        all_kidney_dice.append(dice_score(outputs['kidney_seg'], batch['kidney_mask'].to(device)))
        all_lesion_dice.append(dice_score(outputs['lesion_seg'], batch['lesion_mask'].to(device)))

test_diag_auc = roc_auc_score(all_diag_targets, all_diag_probs)

all_local_probs = np.concatenate(all_local_probs, axis=0)
all_local_targets = np.concatenate(all_local_targets, axis=0)
local_aucs = []
for i in range(NUM_LOCAL_PATHOLOGY_IDS):
    if len(np.unique(all_local_targets[:, i])) > 1:
        local_aucs.append(roc_auc_score(all_local_targets[:, i], all_local_probs[:, i]))

print('PHASE 4 RESULTS (full multi-task scaffold, fixed unweighted loss sum):')
print(f'  Healthy/Pathological AUC   : {test_diag_auc:.4f}   (Phase 2 baseline: 0.9003)')
print(f'  Kidney Segmentation Dice   : {np.mean(all_kidney_dice):.4f}   (Phase 2 baseline: 0.8735)')
print(f'  Lesion (coarse) Seg. Dice  : {np.mean(all_lesion_dice):.4f}   (new -- no prior baseline)')
print(f'  Local Pathology mean AUC   : {np.mean(local_aucs):.4f}   (across {len(local_aucs)} pathology IDs with both classes present)')
print()
print('Phase 4 complete. Next: Phase 5 adds an anatomical-consistency constraint between')
print('the kidney mask and lesion mask predictions.')

NameError: name 'dice_score' is not defined

In [13]:
# Load Phase 4's checkpoint and compute the violation rate it never reported
model.load_state_dict(torch.load(os.path.join(DRIVE_CHECKPOINTS_PATH, 'phase4_multitask_best.pt')))
model.eval()

def consistency_violation_rate(lesion_logits, kidney_mask_target, eps=1e-6):
    lesion_prob = torch.sigmoid(lesion_logits)
    total_mass = lesion_prob.flatten(1).sum(dim=1) + eps
    outside_mass = (lesion_prob * (1.0 - kidney_mask_target)).flatten(1).sum(dim=1)
    return (outside_mass / total_mass).mean().item()

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
all_violation = []
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        outputs = model(images)
        all_violation.append(consistency_violation_rate(outputs['lesion_seg'], batch['kidney_mask'].to(device)))

print(f'Phase 4 (no constraint) violation rate: {np.mean(all_violation):.4f}')
print(f'Phase 5 (with constraint) violation rate: 0.3790')

NameError: name 'BATCH_SIZE' is not defined